
# Consumption–Saving RL — PPO Smoke Test

목적:
- DQN은 discrete saving-share action(5% 단위)을 사용했습니다.
- 이번에는 **continuous saving share \(z \in [0,0.95]\)** 를 직접 학습하는 PPO를 테스트합니다.
- 비교 benchmark는 discrete DP-Matched가 아니라 **DP-Fine**입니다.

이번 단계는 **seed=42 한 번의 smoke test**입니다.
성능이 괜찮은지 확인한 뒤에만 checkpoint / multi-seed 본실험으로 확장합니다.

`Run all`에서는 heavy PPO training이 자동 실행되지 않습니다.


In [ ]:

# ============================================================
# 0. Runtime setup
# ============================================================

import sys
import subprocess
import importlib

try:
    import gymnasium as gym
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gymnasium"]
    )
    import gymnasium as gym

# Python 3.13 / SymPy printing workaround
import sympy
sympy_printing = importlib.import_module("sympy.printing")
if not hasattr(sympy, "printing"):
    sympy.printing = sympy_printing

# Stable-Baselines3
try:
    import stable_baselines3
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "stable-baselines3"]
    )
    import stable_baselines3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gymnasium import spaces

import torch

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

print("Python:", sys.version.split()[0])
print("SymPy:", sympy.__version__)
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)
print("Stable-Baselines3:", stable_baselines3.__version__)


In [ ]:

# ============================================================
# 1. Economic calibration
# ============================================================

R = 1.00985
BETA = 0.9703
SIGMA = 2.0

Y_HIGH = 1.0
Y_LOW = 0.472

P_INCOME = np.array(
    [
        [0.9393, 0.0607],
        [0.3920, 0.6080]
    ],
    dtype=np.float64
)

A_MAX_DP = 4.5

print("Calibration loaded.")


In [ ]:

# ============================================================
# 2. Continuous-action environment for PPO
# ============================================================

class ContinuousConsumptionSavingEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(
        self,
        random_initial_state=True,
        initial_asset=1.0,
        initial_income_state=0,
        R=R,
        beta=BETA,
        sigma=SIGMA,
        y_high=Y_HIGH,
        y_low=Y_LOW,
        transition_matrix=P_INCOME
    ):
        super().__init__()

        self.R = float(R)
        self.beta = float(beta)
        self.sigma = float(sigma)

        self.income_values = np.array(
            [y_high, y_low],
            dtype=np.float64
        )

        self.P = np.asarray(
            transition_matrix,
            dtype=np.float64
        )

        self.random_initial_state = bool(
            random_initial_state
        )

        self.initial_asset = float(
            initial_asset
        )

        self.initial_income_state = int(
            initial_income_state
        )

        # Continuous saving share z in [0, 0.95]
        self.action_space = spaces.Box(
            low=np.array([0.0], dtype=np.float32),
            high=np.array([0.95], dtype=np.float32),
            dtype=np.float32
        )

        # Observation = [asset / 4.5, income]
        self.observation_space = spaces.Box(
            low=np.array(
                [0.0, Y_LOW],
                dtype=np.float32
            ),
            high=np.array(
                [np.inf, Y_HIGH],
                dtype=np.float32
            ),
            dtype=np.float32
        )

        self.asset = None
        self.income_state = None

    def _obs(self):
        return np.array(
            [
                self.asset / 4.5,
                self.income_values[self.income_state]
            ],
            dtype=np.float32
        )

    def _utility(self, c):
        if np.isclose(self.sigma, 1.0):
            return float(np.log(c))

        return float(
            c ** (1.0 - self.sigma)
            / (1.0 - self.sigma)
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        options = options or {}

        if "initial_asset" in options:
            self.asset = float(
                options["initial_asset"]
            )
        elif self.random_initial_state:
            self.asset = float(
                self.np_random.uniform(
                    0.0, 4.5
                )
            )
        else:
            self.asset = self.initial_asset

        if "initial_income_state" in options:
            self.income_state = int(
                options["initial_income_state"]
            )
        elif self.random_initial_state:
            self.income_state = int(
                self.np_random.integers(0, 2)
            )
        else:
            self.income_state = (
                self.initial_income_state
            )

        return self._obs(), {}

    def step(self, action):

        z = float(
            np.clip(
                np.asarray(action).reshape(-1)[0],
                0.0,
                0.95
            )
        )

        income = (
            self.income_values[
                self.income_state
            ]
        )

        cash_on_hand = (
            self.R * self.asset
            + income
        )

        next_asset = (
            z * cash_on_hand
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = self._utility(
            consumption
        )

        next_income_state = int(
            self.np_random.choice(
                2,
                p=self.P[
                    self.income_state
                ]
            )
        )

        self.asset = next_asset
        self.income_state = (
            next_income_state
        )

        # Infinite horizon; TimeLimit wrapper will create
        # computational truncations during PPO training.
        terminated = False
        truncated = False

        return (
            self._obs(),
            reward,
            terminated,
            truncated,
            {}
        )


def make_training_env():
    env = ContinuousConsumptionSavingEnv(
        random_initial_state=True
    )

    # Artificial rollout reset every 200 quarters.
    # This is a truncation, not an economic terminal state.
    env = gym.wrappers.TimeLimit(
        env,
        max_episode_steps=200
    )

    env = Monitor(env)

    return env


print("Continuous PPO environment ready.")


In [ ]:

# ============================================================
# 3. DP-Fine benchmark
# ============================================================

def solve_dp_fine(
    R=R,
    beta=BETA,
    sigma=SIGMA,
    y_high=Y_HIGH,
    y_low=Y_LOW,
    transition_matrix=P_INCOME,
    a_min=0.0,
    a_max=A_MAX_DP,
    n_assets=500,
    tolerance=1e-10,
    max_iterations=5000,
):
    P = np.asarray(
        transition_matrix,
        dtype=np.float64
    )

    income_values = np.array(
        [y_high, y_low],
        dtype=np.float64
    )

    asset_grid = np.linspace(
        a_min,
        a_max,
        n_assets
    )

    V = np.zeros(
        (n_assets, 2),
        dtype=np.float64
    )

    policy_asset = np.zeros_like(V)

    for iteration in range(max_iterations):

        V_new = np.empty_like(V)

        for income_state in range(2):

            income = (
                income_values[
                    income_state
                ]
            )

            cash_on_hand = (
                R * asset_grid
                + income
            )

            consumption = (
                cash_on_hand[:, None]
                - asset_grid[None, :]
            )

            utility = np.full(
                consumption.shape,
                -np.inf,
                dtype=np.float64
            )

            feasible = (
                consumption > 1e-12
            )

            if np.isclose(sigma, 1.0):
                utility[feasible] = (
                    np.log(
                        consumption[feasible]
                    )
                )
            else:
                utility[feasible] = (
                    consumption[feasible]
                    ** (1.0 - sigma)
                    / (1.0 - sigma)
                )

            expected_future = (
                P[income_state, 0]
                * V[:, 0]
                +
                P[income_state, 1]
                * V[:, 1]
            )

            Q = (
                utility
                +
                beta
                * expected_future[
                    None, :
                ]
            )

            best_idx = np.argmax(
                Q,
                axis=1
            )

            V_new[:, income_state] = (
                Q[
                    np.arange(n_assets),
                    best_idx
                ]
            )

            policy_asset[:, income_state] = (
                asset_grid[best_idx]
            )

        diff = np.max(
            np.abs(V_new - V)
        )

        V = V_new

        if diff < tolerance:
            print(
                "DP-Fine converged at",
                iteration + 1,
                "iterations."
            )
            break

    return (
        asset_grid,
        V,
        policy_asset
    )


asset_grid_fine, V_dp_fine, policy_asset_fine = (
    solve_dp_fine()
)

dp_fine_value = np.interp(
    1.0,
    asset_grid_fine,
    V_dp_fine[:, 0]
)

cash_high = (
    R * asset_grid_fine
    + Y_HIGH
)

cash_low = (
    R * asset_grid_fine
    + Y_LOW
)

policy_share_fine = np.column_stack(
    [
        policy_asset_fine[:, 0]
        / cash_high,
        policy_asset_fine[:, 1]
        / cash_low
    ]
)

print(
    "DP-Fine Bellman value at (asset=1, High):",
    dp_fine_value
)


In [ ]:

# ============================================================
# 4. PPO evaluation helpers
# ============================================================

def utility_vectorized(
    consumption,
    sigma=SIGMA
):
    if np.isclose(sigma, 1.0):
        return np.log(consumption)

    return (
        consumption ** (1.0 - sigma)
        /
        (1.0 - sigma)
    )


def evaluate_ppo_vectorized(
    model,
    n_episodes=2000,
    horizon=500,
    seed=10000
):
    rng = np.random.default_rng(seed)

    assets = np.ones(
        n_episodes,
        dtype=np.float64
    )

    income_states = np.zeros(
        n_episodes,
        dtype=int
    )

    total_utility = np.zeros(
        n_episodes,
        dtype=np.float64
    )

    discount = 1.0

    for _ in range(horizon):

        incomes = np.where(
            income_states == 0,
            Y_HIGH,
            Y_LOW
        )

        obs = np.column_stack(
            [
                assets / 4.5,
                incomes
            ]
        ).astype(np.float32)

        actions, _ = model.predict(
            obs,
            deterministic=True
        )

        z = np.clip(
            np.asarray(actions).reshape(-1),
            0.0,
            0.95
        )

        cash_on_hand = (
            R * assets
            + incomes
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = utility_vectorized(
            consumption
        )

        total_utility += (
            discount * reward
        )

        assets = (
            z * cash_on_hand
        )

        p_low = np.where(
            income_states == 0,
            P_INCOME[0, 1],
            P_INCOME[1, 1]
        )

        income_states = (
            rng.random(n_episodes)
            < p_low
        ).astype(int)

        discount *= BETA

    mean = total_utility.mean()
    std = total_utility.std(ddof=1)
    se = std / np.sqrt(n_episodes)
    ci = 1.96 * se

    return {
        "mean_utility": mean,
        "std_utility": std,
        "ci_lower": mean - ci,
        "ci_upper": mean + ci,
        "max_final_asset": assets.max()
    }


def get_ppo_policy_on_grid(
    model,
    asset_grid
):
    policy = np.zeros(
        (len(asset_grid), 2)
    )

    for income_state, income in [
        (0, Y_HIGH),
        (1, Y_LOW)
    ]:

        obs = np.column_stack(
            [
                asset_grid / 4.5,
                np.full(
                    len(asset_grid),
                    income
                )
            ]
        ).astype(np.float32)

        actions, _ = model.predict(
            obs,
            deterministic=True
        )

        policy[:, income_state] = (
            np.clip(
                np.asarray(actions).reshape(-1),
                0.0,
                0.95
            )
        )

    return policy


print("PPO evaluation helpers loaded.")



## Heavy PPO smoke-test cell

**계산부하: 중간~무거움.**

- PPO training: 50,000 timesteps
- seed = 42 only
- 아직 multi-seed 본실험이 아닙니다.
- `Run all`에서는 자동 실행되지 않습니다.

결과가 충분히 좋으면 그다음에 checkpoint / 5-seed 본실험으로 확장합니다.


In [ ]:

RUN_HEAVY_TRAINING = False

if not RUN_HEAVY_TRAINING:

    print(
        "Fast setup complete.\n"
        "Change RUN_HEAVY_TRAINING = True and rerun THIS CELL ONLY."
    )

else:

    train_env = make_training_env()

    ppo_model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=BETA,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,
        verbose=1,
        seed=42,
        device="auto"
    )

    ppo_model.learn(
        total_timesteps=50_000
    )

    ppo_test = evaluate_ppo_vectorized(
        ppo_model,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    ppo_policy = get_ppo_policy_on_grid(
        ppo_model,
        asset_grid_fine
    )

    ppo_error = np.abs(
        ppo_policy
        - policy_share_fine
    )

    print("\n=== PPO SMOKE TEST ===")
    print("Mean utility:", ppo_test["mean_utility"])
    print(
        "95% CI:",
        (
            ppo_test["ci_lower"],
            ppo_test["ci_upper"]
        )
    )
    print("Policy MAE vs DP-Fine:", ppo_error.mean())
    print(
        "Within 5pp:",
        np.mean(
            ppo_error <= 0.05
        )
    )
    print(
        "Within 10pp:",
        np.mean(
            ppo_error <= 0.10
        )
    )
    print(
        "Max final asset:",
        ppo_test["max_final_asset"]
    )

    comparison = pd.DataFrame(
        [
            {
                "Model": "DP-Fine (MC reference)",
                "Mean Utility": -36.241647
            },
            {
                "Model": "Best checkpointed DQN",
                "Mean Utility": -36.492848
            },
            {
                "Model": "Best Double DQN",
                "Mean Utility": -36.524183
            },
            {
                "Model": "PPO 50k smoke test",
                "Mean Utility": ppo_test["mean_utility"]
            }
        ]
    )

    display(comparison)

    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_fine,
        policy_share_fine[:, 0],
        label="DP-Fine: High income"
    )

    plt.plot(
        asset_grid_fine,
        ppo_policy[:, 0],
        "--",
        label="PPO: High income"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("PPO vs DP-Fine: High Income")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))

    plt.plot(
        asset_grid_fine,
        policy_share_fine[:, 1],
        label="DP-Fine: Low income"
    )

    plt.plot(
        asset_grid_fine,
        ppo_policy[:, 1],
        "--",
        label="PPO: Low income"
    )

    plt.xlabel("Current Asset")
    plt.ylabel("Saving Share")
    plt.title("PPO vs DP-Fine: Low Income")
    plt.legend()
    plt.show()
